# Multi-Head Attention

One attention head learns one relation at a time. Eight head learn eight. Heads are free! Take more of them.

## Problem definition

The single self-attention head computes one attention matrix. That matrix captures one kind of relationship - usually the one that minimizes loss on whatever the training signal is. 

## Basic Concept

Multi-head attention splits, attends, concatenates.

### Split

Take `X(shape: (N, d_model))`. Project to `Q, K, V`, each of shape `(N, d_model)`, Reshape to `(N, n_heads, d_head)` where `d_head = d_model / n_heads`. Transponse to `(n_heads, N, d_heads)`.

### Attend in parallel.

Run scaled dot-product attention inside each head. Each head produces `(N, d_head)`. The heads operate on different subspaces of the embedding and never talk during attention computation itself.

### Concatenate and project

Stack heads back to `(N, d_model)` and multiply by a learned output matrix `W_o` of shape `(d_model, d_model)`. `W_o` is where heads get to mix.


### Lineage of Variations

| Variant | Q Heads | K/V Heads | Used by |
|---|---|---|---|
| Multi-head (MHA) | N | N | GPT-2, BERT |
| Multi-query (MQA) | N | 1 | PaLM, Falcon |
| Grouped-query (GQA) | N | G (e.g. N/8) | Llama 2/3, Qwen |
| Multi-head latent (MLA) | N | compressed to low-rank | DeepSeek-V2/V3 |

All four share the same attention math. The only difference is **how many independent K/V tensors you cache and compute at inference time**.

---

#### MHA (baseline)

Each head has its own `W_k`, `W_v`. KV cache size:

$$\text{KV cache} \propto 2 \times N_{\text{heads}} \times d_{\text{head}} \times L$$

**Theory:** different heads attend to different relation types (syntax, coreference, position). Full independence maximizes expressiveness.

---

#### MQA — Multi-Query Attention

**Core change:** all N query heads **share one K and one V**.

```
Head 1 Q ─┐
Head 2 Q ─┼─→ same K, same V
Head N Q ─┘
```

**Theory / motivation:** at inference, **KV cache dominates memory** (grows with sequence length × heads). MQA cuts KV cache by ~`num_heads`× with minimal architecture change — only the K/V projection is shared.

**Tradeoff:** less expressive K/V subspaces → small quality drop. Works when throughput matters more than peak benchmark score (PaLM, Falcon).

---

#### GQA — Grouped-Query Attention

**Core change:** N query heads are split into **G groups**; each group shares one K/V head.

```
G = 2, N = 8:
  Q heads 0-3 → K₀, V₀
  Q heads 4-7 → K₁, V₁
```

**Theory / motivation:** MQA (G=1) is too aggressive; MHA (G=N) caches too much. GQA is the **Pareto middle ground** — KV cache shrinks by `N/G`× while keeping more K/V diversity than MQA.

- GQA with G=1 → MQA
- GQA with G=N → MHA

Llama 2/3 often use G=8. Can be trained directly or **upcycled** from a trained MHA checkpoint by averaging K/V weights within each group.

---

#### MLA — Multi-head Latent Attention (DeepSeek)

**Core change:** do not cache full K/V per head. Instead:

1. Project hidden state into a **low-rank latent** `c` (small dimension `d_c ≪ d_model`)
2. Reconstruct per-head K/V from `c` via small matrices: `k_i = W_i^K c`, `v_i = W_i^V c`
3. **Decouple RoPE:** only a small "rope" slice of Q/K gets positional encoding; the latent `c` stays unrotated and is what you cache

```
cache: c  (shape: L × d_c)     ← small!
reconstruct per head on the fly: k_i, v_i from c
```

**Theory / motivation:** MQA/GQA reduce heads but still cache O(`d_head`) per token per group. MLA compresses the **representation itself** — KV cache scales with `d_c` (e.g. 512-dim latent vs 128×8=1024-dim full KV). The low-rank bottleneck acts as learned compression; per-head `W_i^K`, `W_i^V` recover head-specific subspaces at compute time.

**Tradeoff:** extra projection FLOPs per layer, but **much smaller KV cache** → longer context and faster decode. DeepSeek-V2/V3 show MLA can match MHA quality at fraction of the cache.

---

#### Summary: what each variant optimizes

| Variant | KV cache | Expressiveness | Core idea |
|---|---|---|---|
| MHA | largest | highest | independent K/V per head |
| MQA | smallest (among head-sharing) | lowest | one K/V for all heads |
| GQA | `1/G` of MHA | middle | shared K/V within groups |
| MLA | smallest (low-rank `c`) | high (via per-head reconstruction) | compress what you cache, reconstruct at attend time |

# Build your Own

## Split heads from single-head attention

In [2]:
def split_heads(X, n_heads):
    n, d = X.shape
    d_head = d // n_heads
    return X.reshape(n, n_heads, d_head).transpose(1, 0, 2)

def combine_heads(H):
    h, n, d_head = H.shape
    return H.transpose(1, 0, 2).reshape(n, h * d_head)

## Run scaled dot product attention per head

In [1]:
import numpy as np

def softmax(x, axis):
    return np.exp(x) / np.sum(np.exp(x), axis=axis, keepdims=True)

def mha_forward(X, W_q, W_k, W_v, W_o, n_heads):
    Q = X @ W_q
    K = X @ W_k
    V = X @ W_v
    Q = split_heads(Q, n_heads)
    K = split_heads(K, n_heads)
    V = split_heads(V, n_heads)
    
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(Q.shape[-1])

    weights = softmax(scores, axis=-1)

    out = weights @ V

    concat = combine_heads(out)

    return concat @ W_o, weights

## Grouped-Query Attention Variant

In [ ]:
def gqa_project(X, W, n_kv_heads, n_heads):
    kv = split_heads(X @ W, n_kv_heads)   # (kv_heads, n, d_head)
    repeat = n_heads // n_kv_heads
    return np.repeat(kv, repeat, axis=0)   # (n_heads, n, d_head)

## Pytorch

In [ ]:
import torch.nn as nn

# MHA
mha = nn.MultiheadAttention(embed_dim=6, num_heads=2, batch_first=True)

# GQA
from torch.nn.functional import scaled_dot_product_attention
out = scaled_dot_product_attention(Q, K, V, attn_mask=None, dropout=0.0, is_causal=False)